# 00 · Ambiente, dati e protocollo

**Tesi:** *Valutazione di una strategia di market making mediante dati LOB L2, L3 e L4.*

**Obiettivo.** Una strategia di market making passivo viene valutata con lo stesso codice ma con modelli di esecuzione diversi: tre regole che deducono le esecuzioni dai soli prezzi aggregati (dati L2) e un replay che ricostruisce la coda reale degli ordini (dati L3 e L4).

**Domanda di ricerca.** Quanto cambia la valutazione di una strategia di market making passivo quando le esecuzioni stimate con regole L2 vengono confrontate con un replay order-level che considera la posizione nella coda?

**Domanda secondaria.** Il filtro di bassa volatilità riduce le perdite e il rischio di inventario rispetto alla strategia sempre attiva?

| Dataset | Informazione | Unità | Uso principale | Limite |
|---|---|---|---|---|
| FI-2010 | L2 normalizzato | titolo-giorno | baseline L2 | nessuna coda, P&L non monetario |
| Nasdaq ITCH | L3/MBO | titolo-giorno | risultato principale | ordini nascosti e market impact esclusi |
| Hyperliquid | L4 + trade | ora | robustezza | book diff senza timestamp |

I termini tecnici sono spiegati nel **glossario** in fondo al notebook 06.

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/market_making_tesi").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from market_making_tesi import report as R
from market_making_tesi.report import it, pct
R.setup_plots()
FIGURES = ROOT / "results" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
np.random.seed(0)
import platform
from importlib.metadata import version
environment = {"python": platform.python_version(), "sistema": platform.platform(),
               **{n: version(n) for n in ["numpy", "pandas", "pyarrow", "matplotlib"]}}
(ROOT / "results" / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
display(pd.Series(environment, name="versione").to_frame())

,versione
python,3.14.6
sistema,Windows-11-10.0.26300-SP0
numpy,2.5.4
pandas,3.0.6
pyarrow,26.0.0
matplotlib,3.11.2


## 1. Da dove vengono i numeri di questi notebook

I dati grezzi (circa 33 GB) non sono nel repository. I risultati si ottengono in due passaggi:

1. **dai dati grezzi:** `scripts/download_data.py`, `mm-backtest ingest` e `mm-backtest pipeline` producono i file CSV in `results/`;
2. **nei notebook:** 02, 04, 05 e 06 leggono soltanto quei CSV; 01 e 03 leggono anche i dati intermedi in `data/interim/` per grafici e controlli, e se mancano mostrano i grafici salvati.

La cella seguente indica quali dati sono presenti in questa esecuzione e quando sono stati prodotti i risultati.

In [2]:
import datetime as dt
status = pd.DataFrame({
    "presente": {"data/raw (dati grezzi)": R.data_available(ROOT, "data", "raw"),
                 "data/interim (dati intermedi)": R.data_available(ROOT, "data", "interim"),
                 "results/test.csv (risultati della pipeline)": R.data_available(ROOT, "results", "test.csv")},
})
modified = (ROOT / "results" / "test.csv").stat().st_mtime
display(status)
display(Markdown(f"I risultati di test caricati dai notebook sono stati scritti dalla pipeline il "
                 f"{dt.datetime.fromtimestamp(modified):%d/%m/%Y alle %H:%M}. In questa esecuzione i notebook "
                 "**non rieseguono** la pipeline: caricano i CSV."))

,presente
data/raw (dati grezzi),True
data/interim (dati intermedi),True
results/test.csv (risultati della pipeline),True


I risultati di test caricati dai notebook sono stati scritti dalla pipeline il 16/10/2026 alle 13:31. In questa esecuzione i notebook **non rieseguono** la pipeline: caricano i CSV.

## 2. I dati grezzi sono quelli dichiarati?

Ogni file scaricato è registrato in `data/manifest.json` con fonte, data di accesso e impronta SHA-256: se cambiasse anche un solo byte, l'impronta non corrisponderebbe. Se i dati grezzi sono presenti, la cella ricalcola le impronte (qualche minuto).

In [3]:
import hashlib
manifest = json.loads((ROOT / "data" / "manifest.json").read_text(encoding="utf-8"))

def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 22), b""):
            digest.update(block)
    return digest.hexdigest()

rows = []
for key, entry in sorted(manifest.items()):
    path = ROOT / key
    present = path.exists()
    ok = present and path.stat().st_size == entry["bytes"] and sha256(path) == entry["sha256"]
    rows.append({"dataset": key.split("/")[2], "GB": entry["bytes"] / 1e9, "presenti": present, "verificati": ok,
                 "fonte": entry["source"].split("/")[2], "accesso": entry["accessed"]})
files = pd.DataFrame(rows)
summary = files.groupby("dataset").agg(file=("GB", "size"), GB=("GB", "sum"), presenti=("presenti", "sum"),
                                       verificati=("verificati", "sum"), fonte=("fonte", "first"), accesso=("accesso", "max"))
display(summary.style.format({"GB": "{:.2f}"}))
if files.presenti.any():
    assert files[files.presenti].verificati.all(), "file alterati: rieseguire scripts/download_data.py"
else:
    display(Markdown("*Dati grezzi assenti: verifica degli hash non eseguita.*"))

,file,GB,presenti,verificati,fonte,accesso
dataset,,,,,,
fi2010,10,0.94,10,10,etsin.fairdata.fi,2026-10-14
hyperliquid,12,0.79,12,12,zenodo.org,2026-10-14
itch,7,31.01,7,7,emi.nasdaq.com,2026-10-14


## 3. La strategia

La strategia è volutamente semplice e **identica in tutti gli esperimenti**; cambia solo il modo di decidere se i suoi ordini vengono eseguiti. In sintesi:

1. **Misura la volatilità** del prezzo medio (*midpoint*) su una finestra mobile.
2. **Opera solo se il mercato è calmo**: sopra una soglia cancella gli ordini, chiude l'inventario e aspetta.
3. **Quota una coppia**: un ordine di acquisto al miglior prezzo in acquisto (*bid*) e uno di vendita al miglior prezzo in vendita (*ask*). Se entrambi vengono eseguiti, guadagna lo *spread*.
4. **Chiude il rischio**: se dopo un certo tempo è stato eseguito un solo lato, cancella l'altro e chiude la posizione con un ordine a mercato.

Le sottosezioni 3.1-3.5 definiscono la regola in modo completo: notazione, sequenza delle operazioni, pseudocodice, codice Python effettivamente eseguito e un esempio numerico passo per passo.

| Componente | Riferimento | Cosa se ne riprende |
|---|---|---|
| Guadagno sullo spread, rischio di inventario | Ho e Stoll (1981); Avellaneda e Stoikov (2008) | Il market maker guadagna sulla differenza tra le due quote ma, detenendo inventario, è esposto ai movimenti del prezzo. |
| Filtro di volatilità | Avellaneda e Stoikov (2008) | Il rischio di inventario cresce con la varianza del prezzo. Qui, invece di allargare le quote come nel modello ottimo, si smette di quotare sopra una soglia. |
| Inventario limitato | Guéant, Lehalle e Fernandez-Tapia (2013) | Market making con un limite all'inventario; qui il limite è una sola quantità per lato. |
| Liquidazione a scadenza | Cartea, Jaimungal e Penalva (2015) | Le strategie con orizzonte finito chiudono l'inventario residuo pagando lo spread. |
| Perdita per selezione avversa | Glosten e Milgrom (1985) | Le esecuzioni arrivano spesso quando il prezzo sta per muoversi contro il market maker. |

La strategia **non implementa** il modello di Avellaneda e Stoikov: ne riprende solo l'idea che il rischio di inventario cresce con la volatilità.

### 3.1 Notazione e parametri

Il simulatore legge il feed un **record** alla volta: uno snapshot L2 per FI-2010, un messaggio order-level (dopo averlo applicato al book ricostruito) per ITCH e Hyperliquid. Per il record *t* si indicano con:

| Simbolo | Significato |
|---|---|
| *b*ₜ, *a*ₜ | miglior prezzo in acquisto (best bid) e in vendita (best ask) |
| *m*ₜ = (*b*ₜ + *a*ₜ)/2 | midpoint; *s*ₜ = *a*ₜ − *b*ₜ è lo spread |
| *q*ₜ | inventario (positivo = posizione lunga, negativo = corta) |
| *C*ₜ | cassa; il valore della posizione è *E*ₜ = *C*ₜ + *q*ₜ · *m*ₜ (mark to market al midpoint) |
| *r*ₜ | rendimento del midpoint: logaritmico per ITCH e Hyperliquid, differenza semplice per FI-2010 (prezzi normalizzati) |
| σₜ | deviazione standard campionaria degli ultimi *W* rendimenti; non definita se ne sono disponibili meno di *m* |

| Parametro | Nome nel codice | Ruolo |
|---|---|---|
| *Q* | `quote_size` | quantità di ogni ordine |
| *Q*ₘₐₓ | `max_abs_inventory` | inventario massimo in valore assoluto; nella strategia base *Q*ₘₐₓ = *Q*, quindi al più un lato eseguito alla volta |
| *W*, *m* | `volatility_window`, `volatility_min_periods` | finestra della volatilità (record, oppure secondi per ITCH) e numero minimo di rendimenti |
| Δ | `volatility_sample_ns` | solo ITCH: il midpoint viene campionato alla fine di ogni secondo e σ usa questi rendimenti al secondo |
| σ* | `volatility_threshold` | soglia: quantile α (50%, 75% o 90%) della volatilità mobile osservata nella sessione di calibrazione |
| *T* | `max_order_age_events` / `max_order_age_ns` | durata massima di una coppia (record per FI-2010 e Hyperliquid, nanosecondi per ITCH) |
| *L* | `latency_events` / `latency_ns` | latenza di inserimento e cancellazione; zero nel caso base, crescente nello stress test |
| *f*ₘ, *f*ₜ | `maker_fee_bps`, `taker_fee_bps` | commissioni; a zero durante la simulazione e applicate dopo sui fill, perché non cambiano le decisioni |

La strategia **non usa** la profondità del book oltre il primo livello (tranne il filtro di imbalance delle varianti), né previsioni del prezzo. La tabella seguente riporta i valori usati nel test: quelli fissi vengono dai file `config/*.json`, quelli marcati «validazione» (*W*, α, *T*) sono scelti in validazione e letti da `results/selected_params.json`.

In [4]:
from market_making_tesi.config import StrategyConfig
from market_making_tesi.pipeline import DATASETS, session_list
cfg = {n: StrategyConfig.from_json(ROOT / "config" / f"{n}.json") for n in ["fi2010", "itch", "hyperliquid"]}
chosen = R.load_json(ROOT, "selected_params")
units = {"fi2010": "unità", "itch": "azioni", "hyperliquid": "BTC"}

def duration(name):
    p = chosen[name]["params"]
    if "max_order_age_ns" in p:
        return f"{it(p['max_order_age_ns'] / 1e9, 0)} secondi"
    return f"{it(p['max_order_age_events'], 0)} record"

def window(name):
    w = chosen[name]["params"]["volatility_window"]
    return f"{it(w, 0)} secondi" if cfg[name].volatility_sample_ns else f"{it(w, 0)} record"

display(pd.DataFrame({
    "quantità per lato Q": {n: f"{it(c.quote_size, 2 if c.quote_size < 1 else 0)} {units[n]}" for n, c in cfg.items()},
    "inventario massimo": {n: f"{it(c.max_abs_inventory, 2 if c.max_abs_inventory < 1 else 0)} {units[n]}" for n, c in cfg.items()},
    "tick": {n: f"{c.tick_size:g}".replace(".", ",") for n, c in cfg.items()},
    "rendimento": {n: "differenza" if c.return_type == "diff" else "logaritmico" for n, c in cfg.items()},
    "volatilità misurata": {"fi2010": "per osservazione", "itch": "midpoint ogni secondo",
                            "hyperliquid": "per evento (dati senza orari)"},
    "finestra W (validazione)": {n: window(n) for n in cfg},
    "minimo di rendimenti m": {n: it(c.volatility_min_periods, 0) for n, c in cfg.items()},
    "quantile α della soglia (validazione)": {n: pct(chosen[n]["params"]["vol_quantile"], 0) for n in cfg},
    "durata T della coppia (validazione)": {n: duration(n) for n in cfg},
    "latenza nel caso base": {n: "nessuna" for n in cfg},
}).rename(index=R.DATASET_LABELS).T)

,FI-2010,Nasdaq ITCH,Hyperliquid BTC
quantità per lato Q,1 unità,100 azioni,"0,01 BTC"
inventario massimo,1 unità,100 azioni,"0,01 BTC"
tick,"0,0001","0,01",1
rendimento,differenza,logaritmico,logaritmico
volatilità misurata,per osservazione,midpoint ogni secondo,per evento (dati senza orari)
finestra W (validazione),100 record,120 secondi,5.000 record
minimo di rendimenti m,20,20,100
quantile α della soglia (validazione),90%,50%,50%
durata T della coppia (validazione),200 record,30 secondi,5.000 record
latenza nel caso base,nessuna,nessuna,nessuna


### 3.2 Che cosa succede a ogni record

L'ordine delle operazioni è fisso ed è lo stesso per tutti i modelli di esecuzione. Conta perché un ordine **non può essere eseguito nello stesso record in cui viene inserito**: le esecuzioni si valutano sul record successivo, con i soli dati che sarebbero stati disponibili.

1. **Esecuzioni.** Il modello di fill decide se gli ordini virtuali già nel mercato sono stati eseguiti dal nuovo record: con le regole L2 si confronta lo snapshot con il prezzo dell'ordine; con la coda FIFO si esamina il messaggio **prima** di applicarlo al book (un'esecuzione al prezzo della quota la raggiunge solo se il volume davanti è esaurito). Un fill aggiorna cassa e inventario al prezzo dell'ordine (esecuzione *maker*), senza superare *Q*ₘₐₓ.
2. **Latenza.** Con latenza positiva si attivano gli ordini e le cancellazioni il cui ritardo è trascorso. Un ordine che all'attivazione attraverserebbe il book viene rifiutato (ordine *post-only*); una cancellazione richiesta lascia l'ordine eseguibile finché non ha effetto.
3. **Volatilità.** Si aggiorna σₜ con il nuovo midpoint. La stima è **causale**: usa solo midpoint già osservati.
4. **Controllo del rischio** (in quest'ordine di priorità):
   * se σₜ > σ*: cancella tutti gli ordini e liquida l'inventario (*high_volatility*); nessun'altra azione;
   * se un ordine della coppia ha raggiunto la scadenza *T*: cancella la coppia e liquida l'inventario residuo (*pair_expiry*);
   * se c'è inventario ma nessun ordine aperto né operazioni in attesa (può accadere solo con latenza, quando un lato viene eseguito mentre la sua cancellazione è in volo): liquida (*orphan_inventory*).
5. **Nuove quote.** Solo se tutte le condizioni sono vere: σₜ è definita e non supera σ*, il periodo iniziale di *warmup* è finito, non ci sono ordini aperti né operazioni in attesa, lo spread è positivo e l'inventario è nullo. Allora inserisce **una coppia**: acquisto di *Q* a *b*ₜ e vendita di *Q* a *a*ₜ, con scadenza al record *t* + *T* (o all'istante *t* + *T* per ITCH).
6. **Registrazione.** Si salvano midpoint, σₜ, regime, inventario, cassa ed *E*ₜ.

A fine sessione gli ordini aperti vengono cancellati e l'inventario residuo è liquidato (*end_of_sample*), quindi ogni sessione termina con posizione nulla.

**Liquidazione.** Chiudere l'inventario significa un ordine a mercato (*taker*) al best opposto: si vende |*q*| al best bid se *q* > 0, si compra |*q*| al best ask se *q* < 0. Il costo rispetto al midpoint è mezzo spread per unità. Nello scenario di stress *liquidazione al midpoint* lo stesso fill avviene a *m*ₜ.

**Ciclo di vita di una coppia.** Inserita → (eventuali fill parziali o totali su uno o entrambi i lati) → chiusa in uno di quattro modi: entrambi i lati eseguiti (guadagno pari allo spread di inserimento), scadenza con un solo lato eseguito (lato aperto cancellato e inventario liquidato), scadenza senza esecuzioni (cancellazione), volatilità sopra soglia (cancellazione e liquidazione). Solo dopo la chiusura può nascere la coppia successiva.

### 3.3 Pseudocodice

```text
INPUT   feed di record ordinati nel tempo (snapshot L2 o messaggi order-level)
        modello di esecuzione FILL ∈ {touch, price-through, cross, coda FIFO}
        parametri Q, Qmax, W, m, σ*, T, L, warmup
STATO   q ← 0 (inventario), C ← 0 (cassa), O ← ∅ (ordini virtuali), σ ← indefinita

per ogni record t:

    # 1. esecuzioni: solo ordini entrati nel mercato in un record precedente
    per ogni ordine o ∈ O vivo con t > t_ingresso(o):
        x ← FILL(o, record t)                       # quantità eseguita (0 se nessun fill)
        x ← min(x, residuo(o), spazio residuo sotto Qmax)
        se x > 0:
            se o è un acquisto: q ← q + x ; C ← C − prezzo(o)·x
            se o è una vendita: q ← q − x ; C ← C + prezzo(o)·x
            residuo(o) ← residuo(o) − x             # a zero l'ordine si chiude

    # 2. latenza (solo se L > 0)
    attiva gli ordini con ritardo trascorso; rifiuta quelli che attraverserebbero il book
    applica le cancellazioni con ritardo trascorso; esegui la liquidazione in attesa se dovuta

    # 3. volatilità causale
    r ← rendimento del midpoint (per record; per ITCH ultimo midpoint di ogni secondo)
    σ ← deviazione standard degli ultimi W rendimenti   (indefinita se meno di m)
    alta_vol ← (σ definita) e (σ > σ*)

    # 4. controllo del rischio
    se alta_vol:
        se O ≠ ∅ oppure q ≠ 0:
            CANCELLA(O) ; LIQUIDA("high_volatility")
    altrimenti se esiste o ∈ O con t ≥ scadenza(o):
            CANCELLA(O) ; LIQUIDA("pair_expiry")
    altrimenti se q ≠ 0 e O = ∅ e nessuna operazione in attesa:
            LIQUIDA("orphan_inventory")

    # 5. nuove quote
    se σ definita e non alta_vol e t > warmup e O = ∅ e nessuna operazione in attesa
       e (a_t − b_t) > 0 e q = 0:
        scadenza ← t + T
        se q + Q ≤ Qmax:  INSERISCI acquisto  (prezzo b_t, quantità Q, scadenza)
        se q − Q ≥ −Qmax: INSERISCI vendita   (prezzo a_t, quantità Q, scadenza)

    # 6. registrazione
    E_t ← C + q · m_t

a fine sessione: CANCELLA(O) ; LIQUIDA("end_of_sample")

LIQUIDA(motivo):     se q > 0: vendi q al best bid  ; se q < 0: compra |q| al best ask
                     (fill taker; con stress "midpoint" il prezzo è m_t) ; q ← 0
INSERISCI(o):        con L = 0 l'ordine è subito nel mercato; con L > 0 entra dopo L
                     e viene rifiutato se nel frattempo attraverserebbe il book (post-only)
CANCELLA(O):         con L = 0 immediata; con L > 0 ha effetto dopo L e nel frattempo
                     l'ordine resta eseguibile
```

**Regole di esecuzione** (funzione FILL; *p* è il prezzo dell'ordine; le tre regole L2 eseguono l'intero residuo):

```text
touch          acquisto se b_t ≤ p          vendita se a_t ≥ p
price-through  acquisto se b_t < p          vendita se a_t > p
cross          acquisto se a_t ≤ p          vendita se b_t ≥ p

coda FIFO      all'ingresso:  davanti(o) ← ordini già presenti al prezzo p sullo stesso lato
               per ogni messaggio su un ordine k al prezzo p, stesso lato, PRIMA di applicarlo:
                   se k ∈ davanti(o):
                       cancellazione totale o modifica che aumenta la quantità → togli k
                       esecuzione o cancellazione parziale → riduci la quantità di k
                   altrimenti, se davanti(o) è vuoto e il messaggio è un'esecuzione:
                       fill di o per min(residuo(o), quantità eseguita)
```

Con `touch` un acquisto al best bid risulta eseguito già al record successivo se il best bid non sale: per questo è un **limite superiore ottimistico**. Con la coda FIFO un ordine nuovo si mette in fondo al livello e viene eseguito solo dopo tutto il volume che lo precede.

**Varianti esplorative** (stadio `variants`, non usate per i risultati principali). Le parti modificate rispetto allo pseudocodice base:

```text
inventory skew  (k = skew_ticks, H = max_hold_factor)
    passo 4, scadenza:  CANCELLA(O); liquida solo se q ≠ 0 e la posizione è aperta da almeno H·T
    passo 4, q ≠ 0 senza ordini: liquida solo se la posizione è aperta da almeno H·T
    passo 5, q ≠ 0:     se la posizione è aperta da almeno H·T: LIQUIDA("max_hold")
                        altrimenti INSERISCI un solo ordine passivo che riduce q, quantità |q|:
                          vendita (q > 0) a min(max(a_t − k·tick, b_t + tick), a_t)
                          acquisto (q < 0) a max(min(b_t + k·tick, a_t − tick), b_t)

filtro di imbalance  (θ = imbalance_threshold)
    I_t ← (volume al best bid − volume al best ask) / (somma dei due volumi)
    passo 4: cancella una vendita aperta se I_t > θ, un acquisto aperto se I_t < −θ
    passo 5: non aprire una nuova coppia se |I_t| > θ
```

### 3.4 Il codice Python della strategia

La cella seguente mostra il **codice sorgente effettivamente eseguito** dal backtest, estratto dai moduli del pacchetto: resta quindi sempre allineato a ciò che produce i risultati. La strategia non esegue ordini: restituisce *intenti* (`PlaceOrder`, `CancelAll`, `CancelOrder`, `Liquidate`) che il gestore degli ordini e il portafoglio applicano.

| Funzione | Modulo | Passo |
|---|---|---|
| `MarketMakingStrategy.observe`, `high_volatility` | `strategy.py` | 3 |
| `MarketMakingStrategy.risk` | `strategy.py` | 4 |
| `MarketMakingStrategy.quote`, `_unwind`, `_expired` | `strategy.py` | 5 |
| `Simulator.step` | `engine.py` | sequenza 2-6 |
| `run_l2_backtest`, `run_order_level_backtest` | `engine.py` | ciclo sui record e passo 1 |
| `TouchFillModel`, `PriceThroughFillModel`, `CrossFillModel`, `QueueAwareFillModel` | `execution.py` | regole di esecuzione |
| `Portfolio.liquidate` | `accounting.py` | liquidazione |

In [5]:
import inspect
from market_making_tesi import accounting, engine, execution, strategy

def show_source(*objects):
    for obj in objects:
        module = inspect.getmodule(obj).__name__.split(".")[-1]
        code = inspect.getsource(obj).rstrip()
        display(Markdown(f"**`{module}.py` · `{obj.__qualname__}`**\n\n```python\n{code}\n```"))

show_source(strategy.imbalance, strategy.threatened, strategy.MarketMakingStrategy)

**`strategy.py` · `imbalance`**

```python
def imbalance(snapshot: BookSnapshot) -> float:
    """Imbalance dei volumi al best, in [-1, 1]; positivo = pressione in acquisto."""
    bid, ask = snapshot.bid_sizes[0], snapshot.ask_sizes[0]
    total = bid + ask
    return (bid - ask) / total if total > 0 else 0.0
```

**`strategy.py` · `threatened`**

```python
def threatened(side: Side, value: float, threshold: float) -> bool:
    """Una vendita e minacciata da pressione in acquisto, un acquisto da pressione in vendita."""
    return value > threshold if side is Side.SELL else value < -threshold
```

**`strategy.py` · `MarketMakingStrategy`**

```python
class MarketMakingStrategy:
    def __init__(self, config: StrategyConfig) -> None:
        self.config = config
        self.volatility = VolatilityEstimator(
            window=config.volatility_window,
            min_periods=config.volatility_min_periods,
            method=config.volatility_method,
            return_type=config.return_type,
            ewma_lambda=config.ewma_lambda,
            sample_ns=config.volatility_sample_ns,
        )
        self.ids = count(1)
        self.sigma: float | None = None
        self.open_pair: str | None = None
        self.hold_since: tuple[int, int] | None = None
        self.counters = {"unwind_orders": 0, "imbalance_cancels": 0, "imbalance_skips": 0}

    def observe(self, snapshot: BookSnapshot) -> float | None:
        """Aggiorna la volatilita causale con il midpoint del record."""
        self.sigma = self.volatility.update(snapshot.midpoint, snapshot.timestamp)
        return self.sigma

    @property
    def high_volatility(self) -> bool:
        return self.sigma is not None and self.sigma > self.config.volatility_threshold

    @property
    def skew(self) -> bool:
        return self.config.inventory_mode == "skew"

    def _hold_exceeded(self, snapshot: BookSnapshot) -> bool:
        if self.hold_since is None:
            return False
        config = self.config
        event, timestamp = self.hold_since
        if snapshot.event_index - event >= config.max_hold_factor * config.max_order_age_events:
            return True
        return (config.max_order_age_ns is not None
                and snapshot.timestamp - timestamp >= config.max_hold_factor * config.max_order_age_ns)

    def risk(self, snapshot: BookSnapshot, state: StrategyState) -> list[Intent]:
        """Controlli di rischio: volatilita, imbalance avverso e scadenza delle quote."""
        flat = abs(state.inventory) <= EPS
        if flat:
            self.hold_since = None
        elif self.hold_since is None:
            self.hold_since = (snapshot.event_index, snapshot.timestamp)

        if self.high_volatility:
            if state.working or not flat:
                return [CancelAll("high_volatility"), Liquidate("high_volatility")]
            return []

        intents: list[Intent] = []
        working = state.working
        threshold = self.config.imbalance_threshold
        if threshold is not None and working:
            value = imbalance(snapshot)
            for order in working:
                if threatened(order.side, value, threshold):
                    intents.append(CancelOrder(order.order_id, "imbalance"))
                    self.counters["imbalance_cancels"] += 1
            cancelled = {i.order_id for i in intents}
            working = [o for o in working if o.order_id not in cancelled]

        if any(self._expired(order, snapshot) for order in working):
            intents.append(CancelAll("pair_expiry"))
            if not self.skew:
                intents.append(Liquidate("pair_expiry"))
            elif not flat and self._hold_exceeded(snapshot):
                intents.append(Liquidate("max_hold"))
            return intents
        if not flat and not state.busy and not intents:
            if not self.skew:
                # Inventario senza ordini aperti (fill durante una cancellazione in latenza).
                intents.append(Liquidate("orphan_inventory"))
            elif self._hold_exceeded(snapshot):
                intents.append(Liquidate("max_hold"))
        return intents

    def _common(self, snapshot: BookSnapshot, pair_id: str, size: float) -> dict:
        config = self.config
        return {
            "pair_id": pair_id,
            "symbol": snapshot.symbol,
            "size": size,
            "remaining": size,
            "placed_event": snapshot.event_index,
            "placed_timestamp": snapshot.timestamp,
            "expires_event": snapshot.event_index + config.max_order_age_events,
            "expires_timestamp": (snapshot.timestamp + config.max_order_age_ns
                                  if config.max_order_age_ns is not None else None),
        }

    def quote(self, snapshot: BookSnapshot, state: StrategyState) -> list[Intent]:
        config = self.config
        if (self.sigma is None or self.high_volatility or state.records <= config.warmup_events
                or state.busy or snapshot.spread <= 0):
            return []
        if abs(state.inventory) > EPS:
            return self._unwind(snapshot, state) if self.skew else []
        threshold = config.imbalance_threshold
        if threshold is not None and abs(imbalance(snapshot)) > threshold:
            self.counters["imbalance_skips"] += 1
            return []
        pair_id = f"P{next(self.ids)}"
        self.open_pair = pair_id
        offset = config.quote_offset_ticks * (config.tick_size or 0.0)
        common = self._common(snapshot, pair_id, config.quote_size)
        intents: list[Intent] = []
        # Un lato e quotato solo se un suo fill completo rispetta max_abs_inventory.
        if state.inventory + config.quote_size <= config.max_abs_inventory + EPS:
            intents.append(PlaceOrder(VirtualOrder(order_id=f"{pair_id}-B", side=Side.BUY,
                                                   price=snapshot.best_bid - offset, **common)))
        if state.inventory - config.quote_size >= -config.max_abs_inventory - EPS:
            intents.append(PlaceOrder(VirtualOrder(order_id=f"{pair_id}-S", side=Side.SELL,
                                                   price=snapshot.best_ask + offset, **common)))
        return intents

    def _unwind(self, snapshot: BookSnapshot, state: StrategyState) -> list[Intent]:
        """Quota passiva solo sul lato che riduce l'inventario (inventory skew)."""
        config = self.config
        if self._hold_exceeded(snapshot):
            return [Liquidate("max_hold")]
        side = Side.SELL if state.inventory > 0 else Side.BUY
        threshold = config.imbalance_threshold
        if threshold is not None and threatened(side, imbalance(snapshot), threshold):
            self.counters["imbalance_skips"] += 1
            return []
        improve = config.skew_ticks * (config.tick_size or 0.0)
        if side is Side.SELL:
            price = max(snapshot.best_ask - improve, snapshot.best_bid + (config.tick_size or 0.0))
            price = min(price, snapshot.best_ask)
        else:
            price = min(snapshot.best_bid + improve, snapshot.best_ask - (config.tick_size or 0.0))
            price = max(price, snapshot.best_bid)
        pair_id = self.open_pair or f"P{next(self.ids)}"
        self.counters["unwind_orders"] += 1
        order_id = f"{pair_id}-U{self.counters['unwind_orders']}"
        return [PlaceOrder(VirtualOrder(order_id=order_id, side=side, price=price,
                                        **self._common(snapshot, pair_id, abs(state.inventory))))]

    @staticmethod
    def _expired(order: VirtualOrder, snapshot: BookSnapshot) -> bool:
        if snapshot.event_index >= order.expires_event:
            return True
        return order.expires_timestamp is not None and snapshot.timestamp >= order.expires_timestamp
```

In [6]:
show_source(engine.Simulator.step, engine.run_l2_backtest, engine.run_order_level_backtest)

**`engine.py` · `Simulator.step`**

```python
    def step(self, snapshot: BookSnapshot, book: MarketByOrderBook | None = None,
             fills_applied: bool = False) -> None:
        self.last_snapshot = snapshot
        self.records += 1
        self._resolve_markouts(snapshot.midpoint)
        traded = self._liquidate(snapshot, self.manager.process(snapshot, book)) or fills_applied

        sigma = self.strategy.observe(snapshot)
        if self.strategy.high_volatility and self.records > self.config.warmup_events:
            self.halted_records += 1  # solo nel tratto di valutazione
        risk = self.strategy.risk(snapshot, self._state())
        if any(getattr(intent, "reason", "") == "pair_expiry" for intent in risk):
            self.expired_pairs += 1
        traded = self._liquidate(snapshot, self.manager.submit(risk, snapshot, book)) or traded

        quotes = self.strategy.quote(snapshot, self._state())
        for intent in quotes:
            if not isinstance(intent, PlaceOrder):
                continue
            pair = self.pairs.setdefault(intent.order.pair_id, {
                "placed_event": snapshot.event_index, "size": intent.order.size, "sides": 0,
                "filled_BUY": 0.0, "filled_SELL": 0.0})
            pair["sides"] += 1
        traded = self._liquidate(snapshot, self.manager.submit(quotes, snapshot, book)) or traded
        self.record(snapshot, sigma, force=traded)
```

**`engine.py` · `run_l2_backtest`**

```python
def run_l2_backtest(
    snapshots: Iterable[BookSnapshot],
    config: StrategyConfig,
    fill_model: L2FillModel,
    record_every: int = 1,
    record_fills: bool = True,
    markout_records: int = 100,
) -> BacktestReport:
    """Replay su snapshot L2 con una regola euristica di fill."""
    sim = Simulator(config, record_every, record_fills, markout_records)
    for snapshot in snapshots:
        live = sim.manager.live()
        applied = bool(live) and sim.apply_fills(fill_model.fills(snapshot, live), snapshot.midpoint)
        sim.step(snapshot, fills_applied=applied)
    return sim.finish("L2")
```

**`engine.py` · `run_order_level_backtest`**

```python
def run_order_level_backtest(
    events: Iterable[OrderEvent],
    config: StrategyConfig,
    depth: int = 10,
    record_every: int = 1,
    record_fills: bool = True,
    markout_records: int = 100,
) -> BacktestReport:
    """Replay order-level con coda FIFO.

    Gli eventi con ``metadata["bootstrap"]`` ricostruiscono soltanto il book
    (stato iniziale, pre-apertura) e non attivano la strategia.
    """
    queue_model = QueueAwareFillModel()
    sim = Simulator(config, record_every, record_fills, markout_records, queue_model)
    book = MarketByOrderBook()
    for event in events:
        if event.metadata.get("bootstrap"):
            book.apply(event)
            continue
        live = sim.manager.live()
        applied = False
        if live:
            fills = queue_model.fills_before_apply(event, book.get(event.order_id), live,
                                                   sim.last_snapshot)
            if fills:
                mid = sim.last_snapshot.midpoint if sim.last_snapshot else float("nan")
                applied = sim.apply_fills(fills, mid)
        book.apply(event)
        snapshot = book.snapshot(event.timestamp, event.event_index, depth=depth)
        if snapshot is None:
            continue
        sim.step(snapshot, book, fills_applied=applied)
    return sim.finish("ORDER_LEVEL")
```

In [7]:
show_source(execution.TouchFillModel, execution.PriceThroughFillModel, execution.CrossFillModel,
            execution.QueueAwareFillModel, accounting.Portfolio.liquidate)

**`execution.py` · `TouchFillModel`**

```python
class TouchFillModel:
    """Ipotesi ottimistica: il best sullo stesso lato raggiunge la quota."""
    reason = "l2_touch"

    def reached(self, snapshot: BookSnapshot, order: VirtualOrder) -> bool:
        return (snapshot.best_bid <= order.price if order.side is Side.BUY
                else snapshot.best_ask >= order.price)

    def fills(self, snapshot: BookSnapshot, orders: list[VirtualOrder]) -> list[Fill]:
        return [_fill(o, snapshot.timestamp, snapshot.event_index, o.remaining, self.reason)
                for o in orders if o.active and o.live and o.remaining > 0
                and snapshot.event_index > o.live_event and self.reached(snapshot, o)]
```

**`execution.py` · `PriceThroughFillModel`**

```python
class PriceThroughFillModel(TouchFillModel):
    """Ipotesi piu restrittiva: il best dello stesso lato supera la quota.

    Anche questa e una proxy L2, non una prova di esecuzione osservata.
    """
    reason = "l2_price_through"

    def reached(self, snapshot: BookSnapshot, order: VirtualOrder) -> bool:
        return (snapshot.best_bid < order.price if order.side is Side.BUY
                else snapshot.best_ask > order.price)
```

**`execution.py` · `CrossFillModel`**

```python
class CrossFillModel(TouchFillModel):
    """Proxy sul lato opposto: il best contrario raggiunge la quota.

    Un acquisto e eseguito se il best ask scende fino al prezzo dell'ordine
    (``best_ask <= price``), una vendita se il best bid sale fino ad esso.
    E la convenzione piu diffusa in letteratura per i backtest su snapshot:
    a differenza di ``touch`` non scatta quando il mercato resta fermo.
    """
    reason = "l2_cross"

    def reached(self, snapshot: BookSnapshot, order: VirtualOrder) -> bool:
        return (snapshot.best_ask <= order.price if order.side is Side.BUY
                else snapshot.best_bid >= order.price)
```

**`execution.py` · `QueueAwareFillModel`**

```python
class QueueAwareFillModel:
    """Coda FIFO al prezzo: cancel riduce la coda ma non genera fill.

    Il volume eseguito su ordini posteriori puo raggiungere la quota virtuale.
    Una modifica che aumenta la size perde priorita (ipotesi del prototipo).
    """

    def initialize_queue(self, order: VirtualOrder, book: MarketByOrderBook) -> None:
        # Tutti gli ordini gia presenti al livello hanno priorita temporale.
        order.ahead_orders = {oid: book.orders[oid].remaining
                              for oid in book.levels[order.side].get(order.price, ())}
        order.queue_ahead = sum(order.ahead_orders.values())

    def fills_before_apply(self, event: OrderEvent, before: HistoricalOrder | None,
                           orders: list[VirtualOrder], snapshot: BookSnapshot | None) -> list[Fill]:
        if before is None:
            return []
        fills = []
        available = max(0.0, min(before.remaining, event.size if event.size is not None else before.remaining))
        for order in orders:
            if not order.active or not order.live or event.event_index <= order.live_event:
                continue
            if before.side is not order.side or before.price != order.price:
                continue
            oid = str(event.order_id)
            was_ahead = oid in order.ahead_orders
            if was_ahead:
                previous = order.ahead_orders[oid]
                if event.event_type in {EventType.DELETE, EventType.REPLACE}:
                    order.ahead_orders.pop(oid)
                elif event.event_type is EventType.UPDATE and event.size is not None:
                    if event.size > before.remaining or event.size <= 0:
                        order.ahead_orders.pop(oid)
                    else:
                        order.ahead_orders[oid] = min(previous, event.size)
                elif event.event_type in {EventType.EXECUTE, EventType.TRADE, EventType.CANCEL}:
                    left = max(0.0, previous - available)
                    if left > 1e-12:
                        order.ahead_orders[oid] = left
                    else:
                        order.ahead_orders.pop(oid)
                order.queue_ahead = sum(order.ahead_orders.values())
            if (not was_ahead and order.queue_ahead <= 1e-12
                    and event.event_type in {EventType.EXECUTE, EventType.TRADE}
                    and available > 0):
                size = min(order.remaining, available)
                fills.append(_fill(order, event.timestamp, event.event_index, size, "queue_aware"))
                available -= size
        return fills
```

**`accounting.py` · `Portfolio.liquidate`**

```python
    def liquidate(self, snapshot: BookSnapshot, fee_bps: float,
                  mode: str = "market", reason: str = "end_of_sample") -> None:
        if mode not in {"market", "midpoint"}:
            raise ValueError(f"Modalita di liquidazione sconosciuta: {mode}")
        if abs(self.inventory) <= 1e-12:
            return
        side = Side.SELL if self.inventory > 0 else Side.BUY
        price = snapshot.best_bid if side is Side.SELL else snapshot.best_ask
        if mode == "midpoint":
            price = snapshot.midpoint
        self.apply(Fill(snapshot.timestamp, snapshot.event_index,
                        f"LIQ-{snapshot.event_index}", "LIQUIDATION", snapshot.symbol,
                        side, price, abs(self.inventory), "taker", reason,
                        mid=snapshot.midpoint), fee_bps)
```

### 3.5 Un esempio passo per passo

Per vedere la regola all'opera, la cella seguente esegue il simulatore vero (`Simulator.step`, con regola `cross`) su 13 snapshot costruiti a mano, con tick 0,01, *Q* = *Q*ₘₐₓ = 1, finestra *W* = 5, *m* = 2, soglia σ* = 0,001 e durata *T* = 4 record. Le commissioni sono nulle.

In [8]:
from market_making_tesi.engine import Simulator
from market_making_tesi.execution import CrossFillModel
from market_making_tesi.models import BookSnapshot, Side

demo = StrategyConfig(quote_size=1.0, max_abs_inventory=1.0, volatility_window=5, volatility_min_periods=2,
                      volatility_threshold=0.001, return_type="log", max_order_age_events=4, tick_size=0.01,
                      maker_fee_bps=0.0, taker_fee_bps=0.0)
book = [(99.99, 100.01)] * 3 + [(99.98, 99.99), (99.99, 100.01), (100.01, 100.02), (100.00, 100.02),
                                (99.99, 100.01), (99.99, 100.01), (99.98, 100.00), (100.50, 100.52),
                                (100.50, 100.52), (100.49, 100.51)]
sim, rule = Simulator(demo), CrossFillModel()
side_name = {Side.BUY: "acquisto", Side.SELL: "vendita"}

def describe(fill):
    kind = "maker" if fill.liquidity == "maker" else f"taker, {fill.reason}"
    return f"{side_name[fill.side]} {it(fill.price, 2)} ({kind})"

rows = []
for t, (bid, ask) in enumerate(book, start=1):
    snap = BookSnapshot(t * 10**9, t, "DEMO", (bid,), (10.0,), (ask,), (10.0,))
    done = len(sim.portfolio.fills)
    live = sim.manager.live()
    applied = bool(live) and sim.apply_fills(rule.fills(snap, live), snap.midpoint)  # passo 1
    sim.step(snap, fills_applied=applied)                                             # passi 2-6
    sigma = sim.strategy.sigma
    rows.append({
        "t": t, "bid": it(bid, 2), "ask": it(ask, 2),
        "σ (×10⁻⁴)": "–" if sigma is None else it(sigma * 1e4, 1),
        "regime": "–" if sigma is None else "alta vol." if sim.strategy.high_volatility else "calmo",
        "esecuzioni nel record": "; ".join(describe(f) for f in sim.portfolio.fills[done:]) or "",
        "ordini aperti a fine record": ", ".join(f"{o.order_id} {side_name[o.side]} {it(o.price, 2)}"
                                                for o in sim.manager.working()),
        "inventario": it(sim.portfolio.inventory, 0),
        "valore E": it(sim.portfolio.equity(snap.midpoint), 3),
    })
trace = pd.DataFrame(rows).set_index("t")
display(trace)
demo_report = sim.finish("L2")
print(f"Valore finale: {it(demo_report.equity_curve[-1]['equity'], 2)}")

,bid,ask,σ (×10⁻⁴),regime,esecuzioni nel record,ordini aperti a fine record,inventario,valore E
t,,,,,,,,
1,"99,99","100,01",–,–,,,0,"0,000"
2,"99,99","100,01",–,–,,,0,"0,000"
3,"99,99","100,01","0,0",calmo,,"P1-B acquisto 99,99, P1-S vendita 100,01",0,"0,000"
4,"99,98","99,99","0,9",calmo,"acquisto 99,99 (maker)","P1-S vendita 100,01",1,"-0,005"
5,"99,99","100,01","1,2",calmo,,"P1-S vendita 100,01",1,"0,010"
6,"100,01","100,02","1,3",calmo,"vendita 100,01 (maker)","P2-B acquisto 100,01, P2-S vendita 100,02",0,"0,020"
7,"100,00","100,02","1,3",calmo,,"P2-B acquisto 100,01, P2-S vendita 100,02",0,"0,020"
8,"99,99","100,01","1,4",calmo,"acquisto 100,01 (maker)","P2-S vendita 100,02",1,"0,010"
9,"99,99","100,01","1,2",calmo,,"P2-S vendita 100,02",1,"0,010"


Valore finale: -0,53


Come leggere la tabella:

* **t = 1-2:** meno di *m* = 2 rendimenti, σ non è definita e la strategia non quota.
* **t = 3:** σ = 0 < σ*: entra la coppia P1 (acquisto a 99,99, vendita a 100,01), con scadenza a *t* = 7.
* **t = 4:** il best ask scende a 99,99, al prezzo dell'acquisto: con la regola `cross` l'acquisto è eseguito e l'inventario passa a 1. La vendita resta aperta.
* **t = 6:** il best bid sale a 100,01: anche la vendita è eseguita. La coppia è chiusa con un guadagno di 0,02 (lo spread di inserimento) e nello stesso record parte la coppia P2 ai nuovi best (100,01 e 100,02).
* **t = 8-10:** di P2 viene eseguito solo l'acquisto (t = 8); a *t* = 10 la coppia scade: la vendita viene cancellata e l'inventario è liquidato al best bid 99,98 (−0,03 sulla coppia). Nello stesso record entra P3.
* **t = 11:** il prezzo salta di circa 50 tick. Prima la vendita di P3 a 100,00 risulta eseguita (il mercato le passa sopra: è la **selezione avversa**), poi la volatilità supera la soglia: la strategia cancella l'acquisto e ricompra a 100,52, perdendo 0,52.
* **t = 12-13:** la volatilità resta sopra soglia e la strategia resta ferma.

L'esempio riassume il meccanismo dei risultati: i guadagni arrivano in piccoli incrementi pari allo spread, mentre le perdite arrivano quando il prezzo si muove contro una quota appena eseguita, e la liquidazione paga mezzo spread in più.

## 4. Protocollo: calibrazione, validazione, test

Le sessioni sono divise **in ordine cronologico**, così i parametri non vengono scelti sugli stessi dati su cui si misura il risultato:

| Fase | A cosa serve |
|---|---|
| Calibrazione | stimare le soglie di volatilità |
| Validazione | scegliere finestra, soglia e durata degli ordini tra 12 combinazioni |
| Test | misurare il risultato, senza più modificare nulla |

Una *sessione* è una giornata (FI-2010, ITCH) o un'ora di negoziazione (Hyperliquid); una *titolo-sessione* è un titolo in una sessione.

In [9]:
rows = []
for name, spec in DATASETS.items():
    for session, symbol, _ in session_list(ROOT, spec):
        phase = "calibrazione" if session in spec.calibration else "validazione" if session in spec.validation else "test"
        rows.append({"dataset": R.DATASET_LABELS[name], "fase": phase, "sessione": session, "simbolo": symbol})
sessions = pd.DataFrame(rows)
order = {"calibrazione": 0, "validazione": 1, "test": 2}
table = (sessions.groupby(["dataset", "fase"]).agg(sessioni=("sessione", "nunique"), titolo_sessioni=("sessione", "size"),
                                                    dal=("sessione", "min"), al=("sessione", "max"))
         .reset_index().sort_values(["dataset", "fase"], key=lambda s: s.map(order) if s.name == "fase" else s))
display(table.set_index(["dataset", "fase"]))

sessioni  titolo_sessioni          dal  \
dataset         fase                                                   
FI-2010         calibrazione         1                5        day01   
                validazione          2               10        day02   
                test                 7               35        day04   
Hyperliquid BTC calibrazione         1                1  20251201T08   
                validazione          1                1  20251201T14   
                test                 4                4  20251202T08   
Nasdaq ITCH     calibrazione         1                3   2019-01-30   
                validazione          1                3   2019-03-27   
                test                 5               15   2019-07-30   

                                       al  
dataset         fase                       
FI-2010         calibrazione        day01  
                validazione         day03  
                test                day10  
Hyperliquid BTC calibrazione  20251201T08  
                validazione   20251201T14  
                test          20251203T14  
Nasdaq ITCH     calibrazione   2019-01-30  
                validazione    2019-03-27  
                test           2020-01-30

**Rischio di overfitting.** FI-2010 usa 1 giornata di calibrazione e 2 di validazione, ITCH 1 e 1, Hyperliquid 1 ora e 1 ora. Su un validation set così piccolo il confronto tra 12 combinazioni può premiare quella che funziona per caso (Bailey, Borwein, López de Prado e Zhu, 2017), soprattutto per ITCH e Hyperliquid. Per questo i notebook mostrano l'intera griglia di validazione e il test non viene mai usato per scegliere.

## 5. Il codice funziona?

I test automatici verificano ogni componente su casi costruiti a mano: lettura dei file, ricostruzione del book, regole di esecuzione, coda, latenza, contabilità, metriche e bootstrap per sessione. Poi il motore viene eseguito sui piccoli campioni di `data/samples`, che devono riprodurre valori noti.

In [10]:
import subprocess
check = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests"],
                       cwd=ROOT, capture_output=True, text=True, timeout=600)
log = check.stdout + check.stderr
(ROOT / "results" / "test_report.txt").write_text(log, encoding="utf-8")
assert check.returncode == 0, log
ran = next(line for line in log.splitlines() if line.startswith("Ran "))
from market_making_tesi.experiments import run_samples
samples, _ = run_samples(ROOT, ROOT / "results" / "samples")
assert samples.fill_maker.tolist() == [22, 5, 2, 2, 0]
R.show_box("Esito dei controlli", [
    f"{ran.split()[1]} test automatici superati.",
    "I cinque campioni di controllo riproducono i valori attesi.",
    "Dati grezzi verificati sul manifesto." if files.presenti.any() else "Dati grezzi non presenti: hash non verificati.",
])

> **Esito dei controlli**
>
> - 51 test automatici superati.
> - I cinque campioni di controllo riproducono i valori attesi.
> - Dati grezzi verificati sul manifesto.